In [2]:
import pandas as pd

quality_score = pd.read_csv('../output/sleep_score/sleep_quality_score_by_duration.csv')

quality_score_avg = quality_score.groupby('patient_id')[['scaled_sleep_quality_sum','scaled_sleep_quality_mean']].mean().reset_index()

quality_score_avg

In [3]:
regularity_score = pd.read_csv('../output/sleep_score/sleep_regularity_score.csv')

regularity_score_avg = regularity_score.groupby('patient_id')[['sleep_regularity_cs','sleep_irregularity_dis']].mean().reset_index()

regularity_score_avg

# id 76230 only has 4 data points in sleep so cannot calculate the regularity measure for it

# validate against number of agitation episode

In [4]:
agitation_number = pd.read_csv('../output/data_cleaned/agitation_number.csv')
agitation_number

In [17]:
merged_df = pd.merge(quality_score_avg, regularity_score_avg, on='patient_id')  # First merge
merged_df = pd.merge(agitation_number,merged_df, on='patient_id', how = 'outer')#.fillna(0) 
## changes: exclude participants without sleep data and then fill NA in agitation as 0
merged_df = merged_df.dropna(subset=['scaled_sleep_quality_sum', 'sleep_regularity_cs']).fillna(0)

print(merged_df.shape)
print(merged_df.head(2))
correlation_matrix = merged_df.drop(columns='patient_id').corr(method='kendall')
correlation_matrix.columns = ['Agitation episodes', 'Sleep quality sum', 'Sleep quality mean', 'Regularity cosine similarity', 'Irregularity Euclidean distance']

# Change row names (index)
correlation_matrix.index = ['Agitation episodes', 'Sleep quality sum', 'Sleep quality mean', 'Regularity cosine similarity', 'Irregularity Euclidean distance']

correlation_matrix


In [18]:
merged_df

In [19]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
# Set up the matplotlib figure
plt.figure(figsize=(10, 8))

mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))
# Plot the correlation matrix as a heatmap
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5, vmin=-1, vmax=1, mask =mask )
plt.tick_params(axis='both', which='major', labelsize=14)


# Display the plot
# plt.title('Correlation between agitation and different sleep scores')
plt.xticks(rotation=15, ha='right')
plt.savefig('../output/sleep_score/correlation_agitation_sleep_score.png', dpi=300, bbox_inches='tight')
plt.show()

# validate against age 

In [35]:
demog = pd.read_csv('../Dataset/Demographics.csv')

merged_df = pd.merge(quality_score_avg, regularity_score_avg, on='patient_id')  # First merge
merged_df = pd.merge(demog,merged_df, on='patient_id')
merged_df

In [57]:
merged_df.columns[3:][0]

In [76]:
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
age_order = ['(70, 80]',  '(80, 90]', '(90, 110]']
merged_df['age'] = pd.Categorical(merged_df['age'], categories=age_order, ordered=True)

# Plot each dependent measure in a separate subplot
sns.boxplot(x='age', y=merged_df.columns[3:][0], data=merged_df, ax=axes[0], palette='Set1', 
            boxprops=dict(alpha=0.8))
sns.scatterplot(x='age', y='scaled_sleep_quality_sum', data=merged_df, ax=axes[0], hue='age', palette='Set1', s=100, marker='D', legend=False)
axes[0].set_title(merged_df.columns[3:][0])

sns.boxplot(x='age', y=merged_df.columns[3:][1], data=merged_df, ax=axes[1], palette='Set1', 
            boxprops=dict(alpha=0.8))
sns.scatterplot(x='age', y='scaled_sleep_quality_mean', data=merged_df, ax=axes[1], hue='age', palette='Set1', s=100, marker='D', legend=False)
axes[1].set_title(merged_df.columns[3:][1])

sns.boxplot(x='age', y=merged_df.columns[3:][2], data=merged_df, ax=axes[2], palette='Set1', 
            boxprops=dict(alpha=0.8))
sns.scatterplot(x='age', y='sleep_regularity_cs', data=merged_df, ax=axes[2], hue='age', palette='Set1', s=100, marker='D', legend=False)
axes[2].set_title(merged_df.columns[3:][2])

sns.boxplot(x='age', y=merged_df.columns[3:][3], data=merged_df, ax=axes[3], palette='Set1', 
            boxprops=dict(alpha=0.8))
sns.scatterplot(x='age', y='sleep_irregularity_dis', data=merged_df, ax=axes[3], hue='age', palette='Set1', s=100, marker='D', legend=False)
axes[3].set_title(merged_df.columns[3:][3])

for ax in axes:
    ax.set_ylabel('') 
    
# Adjust the layout and show the plot
plt.tight_layout()

plt.savefig('../output/sleep_score/BarChart_AGE_sleep_score.png', dpi=300, bbox_inches='tight')
plt.show()
# 'Distribution of Scores Across Different Age Groups'